In [47]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
import os
from langchain_core.globals import set_debug


set_debug(True)
load_dotenv()

True

In [48]:
from langchain.tools import tool
import csv

class Transaction(BaseModel):
    date: str
    description: str
    amount: float
    category: str | None = None
@tool
def read_transactions(file_path: str)-> list[dict]:
    """Read a CSV of transactions and return a list of dicts.
    Each row should become {"date": str, "description": str, "amount": float}.
    """
    if not os.path.exists(file_path):
        filename = os.path.basename(file_path)
        if os.path.exists(filename):
            file_path = filename
            
    transactions_records = []
    # Open and safely read the CSV file
    with open(file_path, mode='r', newline='', encoding='utf-8') as file:
        reader = csv.reader(file)
        next(reader, None)
        for row in reader:
            record_dict= {
                "date": row[0],
                "description" : row[1],
                "amount" : row[2]
            }
            transactions_records.append(record_dict)
        return transactions_records

transactions = read_transactions.invoke({"file_path": "./expencs.csv"})
print(transactions)
    

Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')


[tool/end] [tool:read_transactions] s] Exiting Tool run with output:
"[{'date': '2026-08-01', 'description': 'Big Basket Grocery', 'amount': '2400'}, {'date': '2026-08-02', 'description': 'Uber ride to office', 'amount': '320'}, {'date': '2026-08-03', 'description': 'Netflix subscription', 'amount': '649'}, {'date': '2026-08-04', 'description': 'Electricity bill', 'amount': '4100'}, {'date': '2026-08-05', 'description': 'Fancy dinner at ITC', 'amount': '7800'}, {'date': '2026-08-06', 'description': 'Zomato order', 'amount': '540'}, {'date': '2026-08-07', 'description': 'Flight ticket Mumbai', 'amount': '12500'}, {'date': '2026-08-08', 'description': 'Petrol for bike', 'amount': '1200'}]"
[{'date': '2026-08-01', 'description': 'Big Basket Grocery', 'amount': '2400'}, {'date': '2026-08-02', 'description': 'Uber ride to office', 'amount': '320'}, {'date': '2026-08-03', 'description': 'Netflix subscription', 'amount': '649'}, {'date': '2026-08-04', 'description': 'Electricity bill', 'amoun

In [49]:
@tool
def calculate_category_totals(transactions: list[Transaction]) -> dict:
    """Given transactions that each include a 'category' key,
    return {category: total_amount} summed across all transactions.
    """
    totals = {}
    for tx in transactions:
        category = tx.category
        amount = tx.amount

        if category in totals:
            totals[category]+=amount
        else: 
            totals[category]=amount
    return totals


In [50]:
@tool
def flag_unusual_spends(transactions: list[Transaction], threshold: float = 5000) -> list[Transaction]:
    """Return the subset of transactions whose amount exceeds `threshold`."""
    unusual_spends = []
    for tx in transactions:
        if tx.amount > threshold:
            unusual_spends.append(tx)
    return unusual_spends

In [51]:
class TransactionCategory(BaseModel):
    description: str
    category: str

class CategorizationResult(BaseModel):
    categories: list[TransactionCategory] = Field(
        description="One category per transaction, in the same order as the input descriptions"
    )

@tool
def categorize_transactions(transactions: list[Transaction]) -> list[Transaction]:
    """Given a list of transactions (each with date, description, amount),
    assign a sensible category to each one and return the SAME transactions
    with 'category' filled in on each.
    """
    descriptions = [tx.description for tx in transactions]

    categorizer = ChatGoogleGenerativeAI(model="gemini-3.6-flash")
    structured_categorizer = categorizer.with_structured_output(CategorizationResult)

    prompt = (
        "Assign a sensible category (e.g. Groceries, Transport, Entertainment, "
        "Utilities, Dining, Other) to each transaction description below, "
        "preserving the exact order:\n"
        + "\n".join(f"{i+1}. {d}" for i, d in enumerate(descriptions))
    )

    result = structured_categorizer.invoke(prompt)

    if len(result.categories) != len(transactions):
        raise ValueError(
            f"Expected {len(transactions)} categories, got {len(result.categories)}"
        )

    categorized = []
    for tx, cat in zip(transactions, result.categories):
        categorized.append(tx.model_copy(update={"category": cat.category}))
    return categorized

In [52]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
)
class ExpenseReport(BaseModel):
    category_totals: dict[str, float] = Field(description="Total spend per category")
    flagged_transactions: list[dict] = Field(description="Transactions flagged as unusual")
    summary: str = Field(description="2-3 sentence natural language summary of spending patterns")

agent = create_agent(
    model=model,
    tools=[read_transactions, categorize_transactions, calculate_category_totals, flag_unusual_spends],
    system_prompt=(
    "You are a personal finance assistant. Given a transactions file path:\n"
    "1. Call `read_transactions` to get raw transactions.\n"
    "2. Call `categorize_transactions` on the result to get the categorized list "
    "(each transaction will have a 'category' key added).\n"
    "3. Pass the categorized list of transactions to `calculate_category_totals` and `flag_unusual_spends`.\n"
    "4. Return an ExpenseReport."
    ),
    response_format=ExpenseReport,
)

result = agent.invoke({"messages": [("user", "Read my transactions from ./expense_tracker_agent/expenses.csv")]})


[chain/start] [chain:LangGraph] Entering Chain run with input:
{
  "messages": [
    [
      "user",
      "Read my transactions from ./expense_tracker_agent/expenses.csv"
    ]
  ]
}
[chain/start] [chain:LangGraph > chain:model] Entering Chain run with input:
[inputs]
[llm/start] [chain:LangGraph > chain:model > llm:ChatGoogleGenerativeAI] Entering LLM run with input:
{
  "prompts": [
    "System: You are a personal finance assistant. Given a transactions file path:\n1. Call `read_transactions` to get raw transactions.\n2. Call `categorize_transactions` on the result to get the categorized list (each transaction will have a 'category' key added).\n3. Pass the categorized list of transactions to `calculate_category_totals` and `flag_unusual_spends`.\n4. Return an ExpenseReport.\nHuman: Read my transactions from ./expense_tracker_agent/expenses.csv"
  ]
}


Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')


[llm/end] [chain:LangGraph > chain:model > llm:ChatGoogleGenerativeAI] [1.96s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "",
        "generation_info": {
          "finish_reason": "STOP",
          "model_name": "gemini-3.6-flash",
          "safety_ratings": []
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type": "constructor",
          "id": [
            "langchain",
            "schema",
            "messages",
            "AIMessage"
          ],
          "kwargs": {
            "content": [],
            "additional_kwargs": {
              "function_call": {
                "name": "read_transactions",
                "arguments": "{\"file_path\": \"./expense_tracker_agent/expenses.csv\"}"
              },
              "__gemini_function_call_thought_signatures__": {
                "wPqLmpyT": "EscCCsQCARFNMg89STY2Qb/xI1A1ChClXoOBlVo7yb3zavisBbWplZJ1vVqkO7OuulETIwD2bfjBJ6zRhhORSbrMg9lbW2

Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')


[llm/end] [chain:LangGraph > chain:model > llm:ChatGoogleGenerativeAI] [2.05s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "",
        "generation_info": {
          "finish_reason": "STOP",
          "model_name": "gemini-3.6-flash",
          "safety_ratings": []
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type": "constructor",
          "id": [
            "langchain",
            "schema",
            "messages",
            "AIMessage"
          ],
          "kwargs": {
            "content": [],
            "additional_kwargs": {
              "function_call": {
                "name": "categorize_transactions",
                "arguments": "{\"transactions\": [{\"description\": \"Big Basket Grocery\", \"amount\": 2400, \"date\": \"2026-08-01\"}, {\"amount\": 320, \"date\": \"2026-08-02\", \"description\": \"Uber ride to office\"}, {\"date\": \"2026-08-03\", \"amount\": 649, \"description\": \

Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')
Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')


[llm/end] [chain:LangGraph > chain:model > llm:ChatGoogleGenerativeAI] [4.20s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "",
        "generation_info": {
          "finish_reason": "STOP",
          "model_name": "gemini-3.6-flash",
          "safety_ratings": []
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type": "constructor",
          "id": [
            "langchain",
            "schema",
            "messages",
            "AIMessage"
          ],
          "kwargs": {
            "content": [],
            "additional_kwargs": {
              "function_call": {
                "name": "flag_unusual_spends",
                "arguments": "{\"transactions\": [{\"description\": \"Big Basket Grocery\", \"amount\": 2400, \"category\": \"Groceries\", \"date\": \"2026-08-01\"}, {\"amount\": 320, \"date\": \"2026-08-02\", \"description\": \"Uber ride to office\", \"category\": \"Transport\"}, {\"descr

In [53]:
result["structured_response"] 

ExpenseReport(category_totals={'Dining': 8340.0, 'Entertainment': 649.0, 'Groceries': 2400.0, 'Transport': 14020.0, 'Utilities': 4100.0}, flagged_transactions=[{'amount': 7800, 'category': 'Dining', 'date': '2026-08-05', 'description': 'Fancy dinner at ITC'}, {'amount': 12500, 'category': 'Transport', 'date': '2026-08-07', 'description': 'Flight ticket Mumbai'}], summary='Total spending was highest in Transport driven by a flight ticket to Mumbai, followed by Dining. Two high-value transactions exceeding 5000 were flagged, including the flight ticket and a fancy dinner at ITC. Overall expenses reflect significant discretionary spending on travel and luxury dining.')